# GoldWork_Incremental

Incremental Gold processing plus latest and timestamped Volume snapshots.

## Step 1 — Imports and setup

This cell imports the required helpers, switches to the right catalog, makes sure the Gold schema exists, and creates:

* a `gold_run_id`
* a run date string
* a run timestamp string

These are used for tracking and snapshot publishing.


In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from delta.tables import DeltaTable
from datetime import datetime
import uuid


In [0]:
spark.sql("use catalog adb_datatocrunchproject")
spark.sql("create schema if not exists gold_schema")
gold_run_id = str(uuid.uuid4())

run_ts_str = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")

run_date_str = datetime.utcnow().strftime("%Y-%m-%d")

print("Current Gold Run ID:", gold_run_id)
print("Run Timestamp Folder:", run_ts_str)


## Step 2 — Gold control table

This table stores the latest Gold processing state.

It tells Gold:

* which Silver data was processed last time
* how many Gold rows were merged in the last run


In [0]:
spark.sql("""
create table if not exists adb_datatocrunchproject.gold_schema.processing_control (
layer string,
entity_name string,
last_processed_silver_run_id string,
last_processed_silver_run_ts timestamp,
rows_merged bigint,
run_status string,
gold_run_id string,
updated_at timestamp
)
using delta
""")

## Step 3 — Helper functions

This cell defines reusable Gold functions:

* `upsert_to_gold()` merges data into Gold current-state tables
* `get_last_processed_silver_ts()` reads the Gold watermark from the control table
* `upsert_gold_control()` updates Gold control after a successful run


In [0]:
def upsert_to_gold(df_source, target_table, join_key):
    if spark.catalog.tableExists(target_table):
        dt = DeltaTable.forName(spark, target_table)
        (dt.alias("target")
            .merge(df_source.alias("source"), f"target.{join_key} = source.{join_key}")
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute())
    else:
        df_source.write.format("delta").saveAsTable(target_table)

In [0]:
def get_last_processed_silver_ts(entity_name: str):
    ctrl = (
        spark.table("adb_datatocrunchproject.gold_schema.processing_control")
        .filter(
            (F.col("layer") == 'gold') &
            (F.col("entity_name") == entity_name) &
            (F.col("run_status") == 'SUCCESS')
        )
        .orderBy(F.col("updated_at").desc())
        .limit(1)
    )
    
    rows = ctrl.collect()
    if not rows:
        return None
    return rows[0]["last_processed_silver_run_ts"]


In [0]:
def upsert_gold_control(entity_name, last_processed_silver_run_id, last_processed_run_ts, rows_merged):
    ctrl_df = spark.createDataFrame(
        [(
            "gold",
            entity_name,
            last_processed_silver_run_id,
            last_processed_run_ts,
            int(rows_merged),
            "SUCCESS",
            gold_run_id,
            datetime.utcnow()
        )],
        schema="""
            layer string,
            entity_name string,
            last_processed_silver_run_id string,
            last_processed_silver_run_ts timestamp,
            rows_merged bigint,
            run_status string,
            gold_run_id string,
            updated_at timestamp
        """
    )
    dt = DeltaTable.forName(spark, "adb_datatocrunchproject.gold_schema.processing_control")
    (dt.alias("t")
        .merge(ctrl_df.alias("s"), "t.layer = s.layer and t.entity_name = s.entity_name")
        .whenMatchedUpdate(set={
            "last_processed_silver_run_id": "s.last_processed_silver_run_id",
            "last_processed_silver_run_ts": "s.last_processed_silver_run_ts",
            "rows_merged": "s.rows_merged",
            "run_status": "s.run_status",
            "gold_run_id": "s.gold_run_id",
            "updated_at": "s.updated_at"
        })
        .whenNotMatchedInsertAll()
        .execute())

%md
### Step 4 — Read changed Silver rows only

This cell reads the full Silver current-state tables but filters out only the rows that changed since the last Gold run.

This is the starting point for Gold incremental processing.


In [0]:
# Gold checkpoints track Silver processing-control timestamps, not source business
# timestamps. This keeps orders, products, and payments in the same clock domain
# and detects late-arriving source changes that Silver successfully processed.
silver_control_table = "adb_datatocrunchproject.silver_schema.processing_control"

last_orders_ts = get_last_processed_silver_ts("orders")
last_products_ts = get_last_processed_silver_ts("products")
last_payments_ts = get_last_processed_silver_ts("payments")

silver_orders_current = spark.read.table("adb_datatocrunchproject.silver_schema.orders_transformed")
silver_products_current = spark.read.table("adb_datatocrunchproject.silver_schema.products_transformed")
silver_payments_current = spark.read.table("adb_datatocrunchproject.silver_schema.payments_transformed")

def get_unprocessed_silver_run_ids(entity_name, last_gold_control_ts):
    runs = (
        spark.table(silver_control_table)
        .filter(
            (F.col("layer") == "silver") &
            (F.col("entity_name") == entity_name) &
            (F.col("run_status") == "SUCCESS")
        )
    )
    if last_gold_control_ts is not None:
        runs = runs.filter(F.col("updated_at") > F.lit(last_gold_control_ts))
    return [row["silver_run_id"] for row in runs.select("silver_run_id").distinct().collect()]

orders_run_ids = get_unprocessed_silver_run_ids("orders", last_orders_ts)
products_run_ids = get_unprocessed_silver_run_ids("products", last_products_ts)
payments_run_ids = get_unprocessed_silver_run_ids("payments", last_payments_ts)

# On the first Gold run, build from the full current Silver state. Afterwards,
# process current-state records tagged by Silver runs not yet acknowledged by Gold.
changed_orders = (
    silver_orders_current if last_orders_ts is None
    else silver_orders_current.filter(F.col("silver_run_id").isin(orders_run_ids))
)
changed_products = (
    silver_products_current if last_products_ts is None
    else silver_products_current.filter(F.col("silver_run_id").isin(products_run_ids))
)
changed_payments = (
    silver_payments_current if last_payments_ts is None
    else silver_payments_current.filter(F.col("silver_run_id").isin(payments_run_ids))
)

changed_orders_count = changed_orders.count()
changed_products_count = changed_products.count()
changed_payments_count = changed_payments.count()

print(f"Unprocessed Silver run IDs — orders: {len(orders_run_ids)}")
print(f"Unprocessed Silver run IDs — products: {len(products_run_ids)}")
print(f"Unprocessed Silver run IDs — payments: {len(payments_run_ids)}")
print(f"Number of changed orders = {changed_orders_count}")
print(f"Number of changed products = {changed_products_count}")
print(f"Number of changed payments = {changed_payments_count}")

%md
### Step 5 — Find impacted order IDs

Gold is built at **order grain**, so if anything changes in orders, products, or payments, we identify which `order_id` values are impacted.

Only those order IDs are rebuilt in Gold.


In [0]:
impacted_from_orders = changed_orders.select("order_id").distinct()
impacted_from_payments = changed_payments.select("order_id").distinct()
impacted_from_products = (
    changed_products.alias("p")
    .join(silver_orders_current.alias("o"),
          F.col("p.product_id") == F.col("o.product_id"),
          "inner")
    .select(F.col("o.order_id")).distinct()
)

impacted_order_ids = (
    impacted_from_orders
    .union(impacted_from_payments)
    .union(impacted_from_products)
    .distinct()
)

print("impacted_order_ids = ", impacted_order_ids.count())
display(impacted_order_ids.orderBy("order_id"))


%md
### Step 6 — Build Gold delta for impacted orders

This cell joins the impacted orders with the current Silver products and payments tables, derives business columns, and builds the **Gold delta** that will be merged into the Gold current-state table.


In [0]:
# Preserve the pre-update categories so category moves refresh both sides of the move.
gold_table_name = "adb_datatocrunchproject.gold_schema.orders_information"
if spark.catalog.tableExists(gold_table_name):
    previous_impacted_categories = (
        spark.read.table(gold_table_name)
        .join(impacted_order_ids, "order_id", "inner")
        .select("category")
        .filter(F.col("category").isNotNull())
        .distinct()
    )
else:
    previous_impacted_categories = spark.createDataFrame([], "category string")

# Aggregate payments to one row per order before joining. A payment retry must not
# multiply order rows, and only successful/settled amounts count as money paid.
successful_statuses = ["PAID", "SUCCESS", "SUCCEEDED", "SUCCESSFUL", "COMPLETED", "SETTLED"]
payment_rollup = (
    silver_payments_current
    .groupBy("order_id")
    .agg(
        F.sum(
            F.when(
                F.upper(F.trim(F.col("payment_status"))).isin(successful_statuses),
                F.coalesce(F.col("paid_amount").cast("double"), F.lit(0.0))
            ).otherwise(F.lit(0.0))
        ).alias("paid_amount"),
        F.max(F.col("processed_at").cast("timestamp")).alias("latest_payment_processed_at")
    )
)

# Keep the most recent payment ID/status as descriptive fields while keeping the
# paid_amount metric aggregated across all successful payment records.
payment_window = Window.partitionBy("order_id").orderBy(
    F.col("processed_at").cast("timestamp").desc_nulls_last(),
    F.col("payment_id").cast("string").desc_nulls_last()
)
latest_payment = (
    silver_payments_current
    .withColumn("_payment_rank", F.row_number().over(payment_window))
    .filter(F.col("_payment_rank") == 1)
    .select(
        "order_id",
        "payment_id",
        F.col("payment_status").alias("payment_status")
    )
)

impacted_orders = (
    silver_orders_current.alias("o")
    .join(impacted_order_ids.alias("i"), "order_id", "inner")
)

gold_delta = (
    impacted_orders.alias("o")
    .join(
        silver_products_current.alias("p"),
        F.col("o.product_id") == F.col("p.product_id"),
        "left"
    )
    .join(payment_rollup.alias("pr"), F.col("o.order_id") == F.col("pr.order_id"), "left")
    .join(latest_payment.alias("lp"), F.col("o.order_id") == F.col("lp.order_id"), "left")
    .select(
        F.col("o.order_id"),
        F.col("o.customer_id"),
        F.col("p.product_id"),
        F.col("p.product_name"),
        F.col("p.category"),
        F.col("p.price").alias("products_price"),
        F.col("o.order_status"),
        F.col("o.order_amount"),
        F.col("lp.payment_id").alias("payment_id"),
        F.col("lp.payment_status").alias("payment_status"),
        F.coalesce(F.col("pr.paid_amount"), F.lit(0.0)).alias("paid_amount"),
        F.col("o.order_date"),
        F.col("o.order_month"),
        F.col("o.order_year"),
        F.greatest(
            F.col("o.updated_at").cast("timestamp"),
            F.col("p.updated_at").cast("timestamp"),
            F.col("pr.latest_payment_processed_at").cast("timestamp")
        ).alias("gold_update_ts")
    )
    .withColumn(
        "payment_completion_ratio",
        F.when(
            F.col("order_amount") > 0,
            F.col("paid_amount") / F.col("order_amount")
        ).otherwise(F.lit(0.0))
    )
    .withColumn(
        "payment_state",
        F.when(F.col("order_amount").isNull() | (F.col("order_amount") <= 0), "Invalid_order_amount")
        .when(F.col("payment_completion_ratio") == 0, "Unpaid")
        .when(F.col("payment_completion_ratio") < 1, "Partially_paid")
        .when(F.col("payment_completion_ratio") == 1, "Paid")
        .otherwise("Overpaid")
    )
    .withColumn("gold_updated_date", F.to_date(F.col("gold_update_ts")))
    .withColumn("gold_run_id", F.lit(gold_run_id))
)

# The Gold contract is exactly one current row per order.
if gold_delta.groupBy("order_id").count().filter(F.col("count") > 1).limit(1).count() > 0:
    raise ValueError("Gold delta contains more than one row per order_id; refusing ambiguous MERGE.")

impacted_categories = (
    previous_impacted_categories
    .unionByName(
        gold_delta.select("category").filter(F.col("category").isNotNull()).distinct()
    )
    .distinct()
)

print("gold_delta_rows = ", gold_delta.count())
display(gold_delta)

### Step 7 — Merge Gold current-state table

If Gold delta contains rows, this cell merges them into `gold_schema.orders_information`.

If there are no impacted rows, nothing is merged.


In [0]:
if gold_delta.count() > 0:
    upsert_to_gold(gold_delta, "adb_datatocrunchproject.gold_schema.orders_information", "order_id")
else:
    print("No new rows to insert in gold table")


### Step 8 — Maintain Gold SCD Type 2 history

This cell updates the SCD2 history table.

If a current Gold row changes, the old version is closed (`is_current = false`) and a new current version is inserted.



In [0]:
if not spark.catalog.tableExists("adb_datatocrunchproject.gold_schema.orders_information_scd2"):
    spark.sql("""
        create table adb_datatocrunchproject.gold_schema.orders_information_scd2
        using delta
        as
        select *,
            cast(null as timestamp) as valid_from_ts,
            cast(null as timestamp) as valid_to_ts,
            true as is_current
        from adb_datatocrunchproject.gold_schema.orders_information
        where 1 = 0
    """)

if gold_delta.count() > 0:
    gold_delta.createOrReplaceTempView("gold_delta_view")
    spark.sql("""
        merge into adb_datatocrunchproject.gold_schema.orders_information_scd2 t
        using gold_delta_view s
        on t.order_id = s.order_id and t.is_current = true
        when matched and (
            not(t.order_status <=> s.order_status) or
            not(t.order_amount <=> s.order_amount) or
            not(t.paid_amount <=> s.paid_amount) or
            not(t.payment_id <=> s.payment_id) or
            not(t.category <=> s.category) or
            not(t.product_name <=> s.product_name) or
            not(t.products_price <=> s.products_price) or
            not(t.payment_state <=> s.payment_state)
        ) then update set
            t.is_current = false,
            t.valid_to_ts = s.gold_update_ts
    """)

    spark.sql("""
        insert into adb_datatocrunchproject.gold_schema.orders_information_scd2
        select 
            s.*,
            s.gold_update_ts as valid_from_ts,
            cast(null as timestamp) as valid_to_ts,
            true as is_current
        from gold_delta_view s
        left join adb_datatocrunchproject.gold_schema.orders_information_scd2 t
        on s.order_id = t.order_id and t.is_current = true
        where t.order_id is null or (
            not(t.order_status <=> s.order_status) or
            not(t.order_amount <=> s.order_amount) or
            not(t.paid_amount <=> s.paid_amount) or
            not(t.payment_id <=> s.payment_id) or
            not(t.category <=> s.category) or
            not(t.product_name <=> s.product_name) or
            not(t.products_price <=> s.products_price) or
            not(t.payment_state <=> s.payment_state)
        )
    """)


### Step 9 — Update category-level Gold aggregation

This cell recalculates category-level business metrics only for categories impacted in the current run, then merges them into the category performance Gold table.


In [0]:
# Recompute every affected category from the post-merge order-grain Gold table.
# impacted_categories includes both the old category (before a product move) and
# the new category (after the move).
gold_table_name = "adb_datatocrunchproject.gold_schema.orders_information"
category_table_name = "adb_datatocrunchproject.gold_schema.category_performance"

if impacted_categories.limit(1).count() > 0:
    gold_current = spark.read.table(gold_table_name)
    category_orders = (
        gold_current
        .join(impacted_categories, "category", "inner")
        .groupBy("category")
        .agg(
            F.countDistinct("order_id").alias("total_orders"),
            F.sum(
                F.when(F.col("order_amount") > 0, F.col("order_amount"))
                .otherwise(F.lit(0.0))
            ).alias("Gross_Merchandise_Value"),
            F.sum(
                F.when(F.col("paid_amount") > 0, F.col("paid_amount"))
                .otherwise(F.lit(0.0))
            ).alias("Total_Paid_Amount"),
            F.avg(F.col("payment_completion_ratio")).alias("Average_Payment_Completion_Ratio")
        )
    )

    # Failure rate is a payment-grain metric. Calculate it from Silver payments,
    # not from the one-row-per-order Gold table.
    payment_category_metrics = (
        silver_payments_current.alias("py")
        .join(silver_orders_current.alias("o"), "order_id", "inner")
        .join(
            silver_products_current.alias("p"),
            F.col("o.product_id") == F.col("p.product_id"),
            "inner"
        )
        .join(impacted_categories, F.col("p.category") == F.col("category"), "inner")
        .groupBy(F.col("p.category").alias("category"))
        .agg(
            (
                F.sum(
                    F.when(F.upper(F.trim(F.col("py.payment_status"))) == "FAILED", 1)
                    .otherwise(0)
                ) / F.count("*")
            ).alias("Payment_Failure_Rate")
        )
    )

    category_perf_delta = (
        category_orders
        .join(payment_category_metrics, "category", "left")
        .fillna({"Payment_Failure_Rate": 0.0})
    )
    upsert_to_gold(category_perf_delta, category_table_name, "category")

    # If a category has no remaining orders after a product move, remove its
    # obsolete aggregate row instead of leaving stale metrics behind.
    active_categories = (
        gold_current.select("category")
        .filter(F.col("category").isNotNull())
        .distinct()
    )
    stale_categories = (
        impacted_categories
        .join(active_categories, "category", "left_anti")
        .select("category")
        .collect()
    )
    if stale_categories and spark.catalog.tableExists(category_table_name):
        category_delta = DeltaTable.forName(spark, category_table_name)
        for stale in stale_categories:
            category_delta.delete(F.col("category") == F.lit(stale["category"]))

    print("Recomputed impacted categories:", impacted_categories.count())
else:
    print("No impacted categories in this run.")

%md
### Step 10 — Publish Gold snapshots to Volume

This cell writes two kinds of Gold outputs to a Databricks Volume:

* **latest snapshot** $\rightarrow$ overwritten every successful run
* **timestamped historical snapshot** $\rightarrow$ a new folder for each successful run

This is useful for audit, rollback, and teaching demos.


In [0]:
spark.sql("create volume if not exists adb_datatocrunchproject.gold_schema.gold_snapshots_vol")


In [0]:
latest_orders_path = (
    "/Volumes/adb_datatocrunchproject/gold_schema/gold_snapshots_vol/gold_latest/orders_information"
)

latest_category_path = (
    "/Volumes/adb_datatocrunchproject/gold_schema/gold_snapshots_vol/gold_latest/category_performance"
)

historical_orders_path = f"/Volumes/adb_datatocrunchproject/gold_schema/gold_snapshots_vol/gold_snapshots/orders_information/run_date={run_date_str}/run_ts={run_ts_str.replace(':', '-')}"
historical_category_path = f"/Volumes/adb_datatocrunchproject/gold_schema/gold_snapshots_vol/gold_snapshots/category_performance/run_date={run_date_str}/run_ts={run_ts_str.replace(':', '-')}"

spark.read.table("adb_datatocrunchproject.gold_schema.orders_information").write.mode("overwrite").format("parquet").save(latest_orders_path)
spark.read.table("adb_datatocrunchproject.gold_schema.category_performance").write.mode("overwrite").format("parquet").save(latest_category_path)

spark.read.table("adb_datatocrunchproject.gold_schema.orders_information").write.mode("overwrite").format("parquet").save(historical_orders_path)
spark.read.table("adb_datatocrunchproject.gold_schema.category_performance").write.mode("overwrite").format("parquet").save(historical_category_path)

print("Latest Orders Path :", latest_orders_path)
print("Latest Category Path :", latest_category_path)
print("Historical Orders Path :", historical_orders_path)
print("Historical Category Path :", historical_category_path)

%md
### Step 11 — Update Gold control table

This final cell updates the Gold control table using the latest Silver processing metadata and displays the control table for validation.


In [0]:
# Advance Gold checkpoints to the latest successful Silver control record per
# entity. These control timestamps represent Silver processing time, not source
# business timestamps, so late-arriving updates are still discoverable by run ID.
def latest_silver_control_position(entity_name):
    rows = (
        spark.table("adb_datatocrunchproject.silver_schema.processing_control")
        .filter(
            (F.col("layer") == "silver") &
            (F.col("entity_name") == entity_name) &
            (F.col("run_status") == "SUCCESS")
        )
        .orderBy(F.col("updated_at").desc())
        .limit(1)
        .collect()
    )
    if not rows:
        return None, None
    return rows[0]["silver_run_id"], rows[0]["updated_at"]

orders_run_id, orders_control_ts = latest_silver_control_position("orders")
products_run_id, products_control_ts = latest_silver_control_position("products")
payments_run_id, payments_control_ts = latest_silver_control_position("payments")

upsert_gold_control("orders", orders_run_id, orders_control_ts, changed_orders_count)
upsert_gold_control("products", products_run_id, products_control_ts, changed_products_count)
upsert_gold_control("payments", payments_run_id, payments_control_ts, changed_payments_count)

display(
    spark.table("adb_datatocrunchproject.gold_schema.processing_control")
    .filter(F.col("layer") == "gold")
    .orderBy("entity_name")
)